In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
%run ../00-common/05.data-quality-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
from pyspark.sql import functions as F

In [0]:
races_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "raceName": "race_name",
            "date": "race_date"
        })
)

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

In [0]:
dq_checks = [
    # not_null
    {"name": "season_not_null",     "type": "not_null", "column": "season",     "severity": "critical"},
    {"name": "round_not_null",      "type": "not_null", "column": "round",      "severity": "critical"},
    {"name": "race_name_not_null",  "type": "not_null", "column": "race_name",  "severity": "warning"},
    {"name": "circuit_id_not_null", "type": "not_null", "column": "circuit_id", "severity": "warning"},
    {"name": "race_date_not_null",  "type": "not_null", "column": "race_date",  "severity": "warning"},
    # unique
    {"name": "season_round_unique", "type": "unique", "columns": ["season", "round"], "severity": "critical"},
    # min_rows
    {"name": "min_one_row", "type": "min_rows", "threshold": 1, "severity": "warning"},
    # value_range
    {"name": "season_range", "type": "value_range", "column": "season", "min": 1950, "max": 2100, "severity": "warning"},
    {"name": "round_range",  "type": "value_range", "column": "round",  "min": 1,    "max": 30,   "severity": "warning"},
    # referential integrity
    {
        "name":      "circuit_id_ref_circuits",
        "type":      "referential",
        "column":    "circuit_id",
        "ref_table": f"{catalog_name}.{silver_schema}.circuits",
        "ref_column":"circuit_id",
        "severity":  "warning"
    },
]

run_dq_checks(
    df                  = races_final_df, 
    checks              = dq_checks,
    table_name          = silver_table, 
    batch_id            = v_batch_id, 
    raise_on_critical   = True
)

In [0]:
apply_delta_constraints(
    table_name  = silver_table,
    constraints = [
        {"name": "chk_season_not_null",  "condition": "season IS NOT NULL"},
        {"name": "chk_round_not_null",   "condition": "round IS NOT NULL"},
        {"name": "chk_season_range",     "condition": "season BETWEEN 1950 AND 2100"},
        {"name": "chk_round_range",      "condition": "round BETWEEN 1 AND 30"},
    ]
)

In [0]:
write_to_silver(
    input_df = races_final_df,
    target_table = silver_table,
    merge_condition = "t.season = s.season AND t.round = s.round",
    columns_to_update = [
        "race_name",
        "race_date",
        "circuit_id",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)